# Step 3: Reconciliation

**Why an investor cares.** If the data cube does not tie to the management accounts, nothing built on it can be trusted, and the seller's adviser will attack every number that comes out of it. Reconciliation earns the right to do the analysis. It also tests the management accounts themselves: where the invoices and the reported figures disagree, one of them is wrong, and the investor is about to pay a multiple of the reported figure.

This notebook reads the `recon_*` tables built by `sql/04_reconciliation.sql` and saves them to `outputs/tables/` for the databook.

**Decision on the P06 anomalies.** Section 1 compares the three treatments. `excluded` is the only one under which the cube ties to the management accounts' product lines in all 48 months with no residual, so management left these 15 invoices out of reported revenue. The cube now uses `excluded` (`cfg_settings`). The invoices look like sign-entry errors, so reported revenue is probably understated by their absolute value; that is quantified in section 8 and raised as Q11. The decision is D01 in `docs/decisions_log.md`.

**Status labels.** Every month is `tied` (cube equals the reported total with nothing to explain), `explained` (ties once the anomaly invoices are excluded, as management did) or `unexplained` (the reported total differs from the invoices and the data does not say why).

In [1]:
import pandas as pd
from nb_utils import connect, q, md_table, gbp, pct, TABLES

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
con = connect()
print("current anomaly treatment:", q(con, "SELECT anomaly_treatment FROM cfg_settings").iloc[0, 0])
q(con, "SELECT * FROM cfg_materiality")

current anomaly treatment: excluded


,flag_gbp,material_pct,annual_pct
0,1000.0,0.005,0.001


## 1. Anomaly options

Cube net revenue under each P06 treatment, against the management accounts' reported `total_revenue` and against the sum of their four product-line columns.

In [2]:
opt_sum = q(con, "SELECT * FROM recon_options_summary")
opt_sum.to_csv(TABLES / "step3_anomaly_options.csv", index=False)
opt_sum

,treatment,months_tied_to_total,months_tied_to_lines,abs_diff_vs_total,abs_diff_vs_lines,net_vs_total_2022,net_vs_total_2023,net_vs_total_2024,net_vs_total_2025
0,as_reported,33,36,68843.82,13843.82,-5788.75,16381.04,-6573.48,-16862.63
1,flipped,33,36,65234.88,13843.82,5788.75,20618.96,1573.48,-13137.37
2,excluded,44,48,55000.00,0.00,0.00,18500.00,-2500.00,-15000.00


## 2. Monthly and annual revenue reconciliation (current setting)

In [3]:
monthly = q(con, "SELECT * FROM recon_monthly")
annual = q(con, "SELECT * FROM recon_annual")
monthly.to_csv(TABLES / "step3_recon_monthly.csv", index=False)
annual.to_csv(TABLES / "step3_recon_annual.csv", index=False)
display(annual)
display(monthly.groupby("status").size().rename("months").to_frame())
monthly[monthly.status != "tied"][["month", "mgmt_total_revenue", "cube_net", "difference", "difference_pct", "material", "status", "explanation"]]

,year,mgmt_total_revenue,cube_net,difference,difference_pct,material,months_tied,months_explained,months_unexplained,months_flagged,months_material
0,2022,28232151.64,28232151.64,0.0,0.000,False,8,4,0,0,0
1,2023,32174369.95,32192869.95,18500.0,0.057,False,9,2,1,1,1
2,2024,37261398.51,37258898.51,-2500.0,-0.007,False,7,3,2,2,0
3,2025,45771419.35,45756419.35,-15000.0,-0.033,False,9,2,1,1,0


,months
status,
explained,11
tied,33
unexplained,4


,month,mgmt_total_revenue,cube_net,difference,difference_pct,material,status,explanation
2,2022-03-01,2249417.71,2249417.71,0.0,0.000,False,explained,"Ties once 1 negative paid invoice(s) (INV740991, £702.47 as supplied) are ex..."
3,2022-04-01,2327025.28,2327025.28,0.0,0.000,False,explained,"Ties once 1 negative paid invoice(s) (INV751844, £1,136.08 as supplied) are ..."
6,2022-07-01,2392021.50,2392021.50,0.0,0.000,False,explained,"Ties once 1 negative paid invoice(s) (INV717181, £1,030.98 as supplied) are ..."
8,2022-09-01,2404716.28,2404716.28,0.0,0.000,False,explained,"Ties once 1 negative paid invoice(s) (INV722989, £2,919.22 as supplied) are ..."
12,2023-01-01,2553770.31,2553770.31,0.0,0.000,False,explained,"Ties once 1 negative paid invoice(s) (INV754430, £550.00 as supplied) are ex..."
17,2023-06-01,2716929.59,2735429.59,18500.0,0.681,True,unexplained,Cube ties to the sum of the management product lines. The reported total is ...
18,2023-07-01,2681733.32,2681733.32,0.0,0.000,False,explained,"Ties once 2 negative paid invoice(s) (INV710858, INV758329, £1,568.96 as sup..."
26,2024-03-01,3051622.66,3039622.66,-12000.0,-0.393,False,unexplained,Cube ties to the sum of the management product lines. The reported total is ...
27,2024-04-01,3083644.32,3083644.32,0.0,0.000,False,explained,"Ties once 2 negative paid invoice(s) (INV749523, INV757592, £1,054.82 as sup..."
28,2024-05-01,3078701.34,3078701.34,0.0,0.000,False,explained,"Ties once 1 negative paid invoice(s) (INV713623, £692.38 as supplied) are ex..."


### Unexplained items

The annual differences are made up entirely of these months. June 2023 is the only one above the 0.5% monthly materiality threshold.

In [4]:
unexpl = q(con, "SELECT * FROM recon_unexplained_items")
unexpl.to_csv(TABLES / "step3_unexplained_items.csv", index=False)
check = unexpl.groupby("year").amount.sum().reindex(annual.year, fill_value=0).values - annual.difference.values
assert (abs(check) < 0.005).all(), "annual differences do not equal the unexplained items"
unexpl

,month,year,amount,difference_pct,material,explanation
0,2023-06-01,2023,18500.0,0.681,True,Cube ties to the sum of the management product lines. The reported total is ...
1,2024-03-01,2024,-12000.0,-0.393,False,Cube ties to the sum of the management product lines. The reported total is ...
2,2024-11-01,2024,9500.0,0.298,False,Cube ties to the sum of the management product lines. The reported total is ...
3,2025-02-01,2025,-15000.0,-0.395,False,Cube ties to the sum of the management product lines. The reported total is ...


## 3. Product-line reconciliation (current setting)

In [5]:
pl_annual = q(con, "SELECT * FROM recon_product_line_annual")
pl_monthly = q(con, "SELECT * FROM recon_product_line_monthly")
pl_annual.to_csv(TABLES / "step3_recon_product_line_annual.csv", index=False)
pl_monthly.to_csv(TABLES / "step3_recon_product_line_monthly.csv", index=False)
pl_annual.pivot(index="product_line", columns="year", values="difference")

year,2022,2023,2024,2025
product_line,,,,
Analytics Add-on,0.0,0.0,0.0,0.0
Core Platform,0.0,0.0,0.0,0.0
Implementation Services,0.0,0.0,0.0,0.0
Payments Module,0.0,0.0,0.0,0.0


## 4. Management accounts: internal consistency

In [6]:
internal = q(con, "SELECT * FROM recon_mgmt_internal")
internal.to_csv(TABLES / "step3_mgmt_internal.csv", index=False)
internal[internal.lines_total_mismatch_flag | (internal.gp_arithmetic_diff.abs() > 0.005) | (internal.margin_pct_diff.abs() > 0.00005)]

,month,revenue_sum_of_lines,total_revenue,lines_less_total,gp_arithmetic_diff,margin_pct_diff,lines_total_mismatch_flag
17,2023-06-01,2735429.59,2716929.59,18500.0,0.0,0.0,True
26,2024-03-01,3039622.66,3051622.66,-12000.0,0.0,0.0,True
34,2024-11-01,3200376.09,3190876.09,9500.0,0.0,0.0,True
37,2025-02-01,3787073.54,3802073.54,-15000.0,0.0,0.0,True


## 5. Costs against management cost of sales

In [7]:
costs = q(con, "SELECT * FROM recon_costs_monthly")
costs.to_csv(TABLES / "step3_recon_costs.csv", index=False)
costs_y = costs.assign(year=costs.month.dt.year).groupby("year")[["cost_of_delivery", "cost_of_sales", "difference"]].sum()
print("months with a cost difference:", int((costs.difference.abs() > 0.005).sum()))
costs_y

months with a cost difference: 0


,cost_of_delivery,cost_of_sales,difference
year,,,
2022,6367072.32,6367072.32,0.0
2023,7276948.58,7276948.58,0.0
2024,8276093.15,8276093.15,0.0
2025,12008133.71,12008133.71,0.0


## 6. Source-to-reported walk (current setting)

Each line is a formula of the lines above it: cube net = raw gross billings + less duplicates + anomaly treatment + credit notes; management total = cube net + unreconciled difference.

In [8]:
walk = q(con, "SELECT * FROM recon_walk")
walk["check_cube"] = (walk.raw_gross_billings + walk.less_duplicates + walk.anomaly_treatment + walk.credit_notes - walk.cube_net).abs() < 0.005
walk["check_mgmt"] = (walk.cube_net + walk.unreconciled_difference - walk.mgmt_total).abs() < 0.005
walk.to_csv(TABLES / "step3_walk.csv", index=False)
walk

,year,raw_gross_billings,less_duplicates,anomaly_treatment,credit_notes,cube_net,unreconciled_difference,mgmt_total,check_cube,check_mgmt
0,2022,28564164.39,-8596.92,5788.75,-329204.58,28232151.64,0.0,28232151.64,True,True
1,2023,32537962.92,-70158.24,2118.96,-277053.69,32192869.95,-18500.0,32174369.95,True,True
2,2024,37583315.05,-7115.70,4073.48,-321374.32,37258898.51,2500.0,37261398.51,True,True
3,2025,46174479.60,-11086.40,1862.63,-408836.48,45756419.35,15000.0,45771419.35,True,True


## 7. Flagged months: drill-down

Every month with an absolute difference above the flag threshold, with what the data holds in that month. `residual_after_anomaly_and_line_gap` is the difference left after removing the anomaly amount carried in the cube and the management accounts' own line-sum gap.

In [9]:
drill = q(con, "SELECT * FROM recon_drilldown")
drill.to_csv(TABLES / "step3_drilldown.csv", index=False)
drill

,month,difference,anomaly_amount_in_month,anomaly_rows_in_month,duplicates_removed_in_month,credit_notes_in_month,mgmt_lines_less_total,product_line_differences,residual_after_anomaly_and_line_gap
0,2023-06-01,18500.0,0.00,0,460.62,-16855.72,18500.0,None,0.0
1,2024-03-01,-12000.0,-1804.47,2,524.35,-26943.15,-12000.0,None,0.0
2,2024-11-01,9500.0,0.00,0,0.00,-35951.25,9500.0,None,0.0
3,2025-02-01,-15000.0,0.00,0,2325.73,-31394.01,-15000.0,None,0.0


## 8. Effect on each year's reported total

`manual_adjustments_in_total` is the reported total less the sum of the product lines (the unexplained items with the sign reversed: positive means the reported total is higher than the invoices support). `probable_understatement_from_anomalies` is the absolute value of the 15 excluded invoices, which would add to revenue if they are sign errors.

In [10]:
effects = q(con, "SELECT * FROM recon_reported_total_effects")
effects.to_csv(TABLES / "step3_reported_total_effects.csv", index=False)
effects

,year,manual_adjustments_in_total,probable_understatement_from_anomalies,mgmt_total_revenue,mgmt_sum_of_lines
0,2022,0.0,5788.75,28232151.64,28232151.64
1,2023,-18500.0,2118.96,32174369.95,32192869.95
2,2024,2500.0,4073.48,37261398.51,37258898.51
3,2025,15000.0,1862.63,45771419.35,45756419.35


In [11]:
qa3 = q(con, "SELECT qa_id, topic, question FROM qa_log WHERE step = 'Step 3' ORDER BY qa_id")
qa3

,qa_id,topic,question
0,Q11,Negative invoices,Why were the negative amounts on invoices marked paid left out of reported r...
1,Q12,Management accounts,Please provide the journal listing behind the four manual differences betwee...
2,Q13,Management accounts,"What is the policy on manual adjustments to reported revenue, who can approv..."
3,Q14,Management accounts,Which 2025 revenue figure appears in the information memorandum: the reporte...
